# 2.13-MNIST手写数字识别实践

MNIST 手写数字分类将一张灰度图映射为 0–9 中的一个类别。输入图像先展平为 784 个像素，再依次经过 **128、256 个单元的 sigmoid 隐藏层**，最后输出 10 个类别分数。

实验使用 Adam 优化器，学习率为 0.001，批量大小为 64，默认在 CPU 上训练 3 个完整 epoch。

## 2.13.1-实验目标与流程


| 阶段 | 主要操作 | 检查结果 |
| --- | --- | --- |
| 数据准备 | 读取图像、转换张量、划分训练与验证样本 | 图像形状、标签范围、分区是否重叠 |
| 批量加载 | 训练样本打乱后组成批次 | 每批样本数、数据类型和设备 |
| 模型训练 | 前向计算、交叉熵、反向传播、Adam 更新 | 按样本汇总的训练损失和准确率 |
| 模型选择 | 每轮评估验证集 | 验证损失最低时的权重 |
| 保存与恢复 | 保存 `state_dict`，创建同结构模型后加载 | 恢复前后的预测是否一致 |
| 最终评价 | 遍历测试集并观察误分类 | 测试损失、准确率、错误图像 |

训练集提供梯度，验证集用于选择模型，测试集用于最后一次评价。分开这三种用途，才能区分模型对已学习数据的拟合能力与对未参与选择的数据的表现。

## 2.13.2-环境与实验配置

默认 `REQUESTED_DEVICE='cpu'`。可改为 `auto`，按 CUDA、MPS、CPU 顺序选择可用设备，或明确指定 `cuda` / `mps`。
默认 `num_workers=0`，便于 Windows/Jupyter 直接运行。CPU 线程上限设为 4，避免小网络过度创建线程。

随机种子固定模型初始化、数据划分和训练打乱顺序，但不同 PyTorch 版本、设备或并行算法之间仍可能有数值差异。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import platform
import random
import time

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets
from torchvision.transforms import ToTensor

SEED = 42
BATCH_SIZE = 64
EPOCHS = int(os.environ.get('PYTORCH_NOTES_EPOCHS', '3'))
LEARNING_RATE = 0.001
VALIDATION_SIZE = 5_000
REQUESTED_DEVICE = os.environ.get('PYTORCH_NOTES_DEVICE', 'cpu')
DATA_DIR_OVERRIDE = os.environ.get('PYTORCH_NOTES_DATA_DIR', '').strip()
assert EPOCHS >= 1, 'EPOCHS 必须至少为 1'

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def select_device(requested):
    if requested == 'auto':
        requested = 'cuda' if torch.cuda.is_available() else (
            'mps' if torch.backends.mps.is_available() else 'cpu'
        )
    if requested not in {'cpu', 'cuda', 'mps'}:
        raise ValueError('设备请选择 cpu、cuda、mps 或 auto')
    if requested == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('当前环境没有可用的 CUDA，请改用 cpu 或安装匹配的 PyTorch')
    if requested == 'mps' and not torch.backends.mps.is_available():
        raise RuntimeError('当前环境没有可用的 MPS，请改用 cpu')
    return torch.device(requested)

seed_everything(SEED)
torch.set_num_threads(min(4, os.cpu_count() or 1))
device = select_device(REQUESTED_DEVICE)
versions = {
    'python': platform.python_version(), 'torch': torch.__version__,
    'torchvision': torchvision.__version__, 'numpy': np.__version__,
    'matplotlib': matplotlib.__version__,
}
print(versions)
print('device:', device, '| epochs:', EPOCHS, '| batch_size:', BATCH_SIZE)

## 2.13.3-设置数据与输出目录

本实验以 Jupyter 内核的当前工作目录为 `WORK_DIR`。
该 Notebook 不依赖其他章节文件，运行前需准备所需依赖。下面的代码会打印工作目录和数据路径。更换工作目录后，默认的数据缓存和输出位置会随之改变。

默认缓存根目录是 `WORK_DIR/data/`，四个已解压的 MNIST 文件应位于 `data/MNIST/raw/`：

| 文件 | 内容 |
| --- | --- |
| `train-images-idx3-ubyte` | 60,000 张训练图像 |
| `train-labels-idx1-ubyte` | 训练图像对应的类别标签 |
| `t10k-images-idx3-ubyte` | 10,000 张测试图像 |
| `t10k-labels-idx1-ubyte` | 测试图像对应的类别标签 |

已有缓存时，可以在启动 Jupyter 前设置环境变量 `PYTORCH_NOTES_DATA_DIR`。变量指向包含 `MNIST/raw/` 的缓存根目录，而不是 `raw/` 本身。
绝对路径直接使用，`~` 会展开为用户目录；相对路径统一相对 `WORK_DIR` 解析。未设置或设置为空时使用默认 `data/`。

四个文件均存在时，代码设置 `download=False`。缺少任一文件时，设置 `download=True`，由 TorchVision 检查、下载并解压所需资源；已有可用压缩包可能直接复用，缺少资源时需要联网。
只有 `.gz` 文件不满足这里的已解压缓存检查。存在性检查也不能保证文件内容完好，损坏的缓存仍可能在读取时失败。

每次运行的权重、JSON 和图片保存在 `WORK_DIR/artifacts/mnist-mlp-时间戳/` 子目录，重复运行会创建新目录。
数据缓存目录会在下载时自动创建，输出目录由下面的代码自动创建，无需预先建立。

In [ ]:
WORK_DIR = Path.cwd().resolve()
if DATA_DIR_OVERRIDE:
    DATA_ROOT = Path(DATA_DIR_OVERRIDE).expanduser()
    if not DATA_ROOT.is_absolute():
        DATA_ROOT = WORK_DIR / DATA_ROOT
    DATA_ROOT = DATA_ROOT.resolve()
    DATA_SOURCE = 'custom'
else:
    DATA_ROOT = WORK_DIR / 'data'
    DATA_SOURCE = 'working_directory'

run_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUN_DIR = WORK_DIR / 'artifacts' / f'mnist-mlp-{run_stamp}'
RUN_DIR.mkdir(parents=True, exist_ok=False)
print('工作目录:', WORK_DIR)
print('数据来源:', DATA_SOURCE, '| 缓存根目录:', DATA_ROOT)
print('运行产物:', RUN_DIR.relative_to(WORK_DIR))

## 2.13.4-MNIST 与预处理

MNIST 有 60,000 张训练图、10,000 张测试图，每张是 $28\times28$ 的灰度图，标签为数字 0–9。
`train=True` 读取官方训练分区，`train=False` 读取官方测试分区；它与 `model.train()` 的模型模式不是同一件事。

`ToTensor()` 将此数据集的 PIL 灰度图转为形状 `[1, 28, 28]` 的浮点张量，并将像素从 `[0, 255]` 缩放到 `[0, 1]`。
像素缩放后直接输入模型，不额外加入按均值、标准差进行的标准化或数据增强。测试集不参与训练和模型选择，本例在选定模型后再进行测试。

In [ ]:
raw_filenames = (
    'train-images-idx3-ubyte', 'train-labels-idx1-ubyte',
    't10k-images-idx3-ubyte', 't10k-labels-idx1-ubyte',
)
cache_ready = all((DATA_ROOT / 'MNIST' / 'raw' / name).is_file() for name in raw_filenames)
if not cache_ready:
    print('MNIST 已解压缓存不完整，将启用下载与解压；缺少资源时需要网络。')

full_training_data = datasets.MNIST(
    root=str(DATA_ROOT), train=True, download=not cache_ready, transform=ToTensor(),
)
image, label = full_training_data[0]
assert len(full_training_data) == 60_000
assert image.shape == (1, 28, 28) and image.dtype == torch.float32
assert 0 <= image.min().item() <= image.max().item() <= 1
assert 0 <= label <= 9
print('官方训练分区:', len(full_training_data))
print('单张图:', tuple(image.shape), image.dtype, '| 标签:', label)

## 2.13.5-从训练分区划出独立验证集

从官方训练分区固定随机划分 **55,000 张训练 + 5,000 张验证**。
训练集用于梯度更新；验证集用于每轮监控与选择权重；最终 10,000 张测试图不参加参数或超参数选择。
这里是随机划分，不是按类别分层划分；固定种子和记录索引使本次划分可复查。

如果根据测试结果反复调学习率或换网络，测试集也参与了选择，所得数字就不再是严格独立评价。
尝试改进时只比较验证指标，确定方案后再做最终测试。

In [ ]:
train_size = len(full_training_data) - VALIDATION_SIZE
split_generator = torch.Generator().manual_seed(SEED)
training_data, validation_data = random_split(
    full_training_data, [train_size, VALIDATION_SIZE], generator=split_generator,
)
assert set(training_data.indices).isdisjoint(validation_data.indices)
assert len(training_data) + len(validation_data) == 60_000

split_record = {'seed': SEED, 'train_indices': training_data.indices, 'validation_indices': validation_data.indices}
(RUN_DIR / 'split-indices.json').write_text(json.dumps(split_record), encoding='utf-8')
print('训练:', len(training_data), '| 验证:', len(validation_data))

fig, axes = plt.subplots(3, 3, figsize=(6, 6))
for index, ax in enumerate(axes.flat):
    sample, target = training_data[index]
    ax.imshow(sample.squeeze(0), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'Label: {target}')
    ax.axis('off')
fig.suptitle('MNIST training samples')
fig.tight_layout()
fig.savefig(RUN_DIR / 'training-samples.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

## 2.13.6-DataLoader 将样本组织为小批次

`Dataset` 定义如何取一条样本；`DataLoader` 负责采样和组批。
训练设 `shuffle=True`，每轮重新打乱顺序；验证不打乱。为训练集打乱单独设置随机数生成器，使打乱顺序可以复现。

本实验中 `X` 是 `[N, 1, 28, 28]` 的 `float32`，`y` 是 `[N]` 的 `int64`。
`N` 通常是 64，最后一批可能更小，所以训练代码不能将批量数写死为 64。
本实验每处理一个批次，就更新一次模型参数，记为一个 **step**；完整遍历训练集一次是一个 **epoch**。

例如，55,000 条训练样本按每批 64 条加载，会产生 $\lceil 55000/64\rceil=860$ 个批次，最后一批为 24 条。`drop_last=False` 保留这一小批次，损失统计时仍按实际样本数加权。

In [ ]:
train_generator = torch.Generator().manual_seed(SEED + 1)
train_loader = DataLoader(
    training_data, batch_size=BATCH_SIZE, shuffle=True,
    generator=train_generator, num_workers=0, drop_last=False,
)
validation_loader = DataLoader(
    validation_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
)

# 用验证批次检查接口，避免提前推进训练打乱生成器的状态。
X_example, y_example = next(iter(validation_loader))
print('X:', tuple(X_example.shape), X_example.dtype)
print('y:', tuple(y_example.shape), y_example.dtype)
print('每轮训练批数:', len(train_loader))
assert X_example.shape[1:] == (1, 28, 28)
assert y_example.dtype == torch.int64

## 2.13.7-多层感知机的结构与参数量

| 层 | 输出形状 | 参数个数 |
| --- | --- | --- |
| 输入 | `[N, 1, 28, 28]` | 0 |
| Flatten | `[N, 784]` | 0 |
| Linear + sigmoid | `[N, 128]` | $784\times128+128=100,480$ |
| Linear + sigmoid | `[N, 256]` | $128\times256+256=33,024$ |
| Linear | `[N, 10]` | $256\times10+10=2,570$ |

合计 **136,074 个可训练参数**。MLP 将图像展平后处理，没有利用卷积的局部空间结构。
最后一层输出未归一化 logits；不再接 sigmoid 或 softmax，直接交给多分类交叉熵。

`__init__` 中注册层和参数，`forward` 指定数据流，`model(X)` 执行前向过程。
输入和模型必须放在同一设备。这里在构造模型前重新设种子，让参数初始化不受前面探索单元的随机操作影响。


每个线性层计算仿射变换，sigmoid 再引入非线性：

$$
\sigma(z)=\frac{1}{1+e^{-z}},\qquad
h_1=\sigma(XW_1^T+b_1),\quad
h_2=\sigma(h_1W_2^T+b_2),\quad
Z=h_2W_3^T+b_3.
$$

这里每个偏置向量沿批次轴广播，所有样本共享相同的权重和偏置。

如果去掉两个激活函数，多个含偏置的线性层仍可合并为一次仿射变换，无法利用隐藏层表达更复杂的非线性关系。sigmoid 的输出在 $(0,1)$ 内，当输入绝对值较大时梯度会变小；增加隐藏层并不保证更容易训练。

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.hidden1 = nn.Linear(28 * 28, 128)
        self.hidden2 = nn.Linear(128, 256)
        self.out = nn.Linear(256, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = torch.sigmoid(self.hidden1(x))
        x = torch.sigmoid(self.hidden2(x))
        return self.out(x)

seed_everything(SEED)
model = NeuralNetwork().to(device)
parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert parameter_count == 136_074
with torch.no_grad():
    example_logits = model(X_example.to(device))
assert example_logits.shape == (len(X_example), 10)
print(model)
print('可训练参数:', parameter_count, '| 输出:', tuple(example_logits.shape))

## 2.13.8-定义损失、优化器与按样本统计的循环

损失为默认 `reduction='mean'` 的交叉熵，输入 `[N, 10]` logits 和 `[N]` 整数类别。
优化器使用 `Adam(model.parameters(), lr=0.001)`。本实验没有类别权重或被忽略的标签。

每批先 `zero_grad(set_to_none=True)` 清除旧梯度，再前向、计算损失、反向传播、更新参数。
训练模式用 `model.train()`；验证用 `model.eval()` 加 `inference_mode()`，既控制层行为，也禁用求导。

各批样本数不同时，应按样本数加权，不能直接平均各批的损失。设第 $b$ 批平均损失为 $L_b$，样本数为 $n_b$，则：

$$L_{\mathrm{epoch}}=\frac{\sum_b n_bL_b}{\sum_b n_b},\qquad
\mathrm{accuracy}=\frac{\text{预测正确总数}}{\text{样本总数}}.$$

训练统计来自一轮中持续更新的模型；验证统计来自本轮结束后的固定权重，二者并非完全相同的测量条件。
测试时可顺便收集前 9 个错误样本，用于结果展示。所有累计都转为 Python 数值，避免保留计算图。

假设某次评估只有两批，分别包含 64 和 24 条样本，批平均损失为 0.2 和 0.5。总体平均损失应为 $(64\times0.2+24\times0.5)/88\approx0.282$，而不是 $(0.2+0.5)/2=0.35$；后者会让较小批次获得过高权重。

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

def train_one_epoch(dataloader, model, loss_fn, optimizer, device):
    model.train()
    loss_sum, correct, sample_count = 0.0, 0, 0
    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(X)
        loss = loss_fn(logits, y)
        if not torch.isfinite(loss).item():
            raise RuntimeError('损失出现 NaN 或 Inf，请检查学习率和数据')
        loss.backward()
        optimizer.step()

        batch_count = y.size(0)
        loss_sum += loss.item() * batch_count
        correct += (logits.detach().argmax(dim=1) == y).sum().item()
        sample_count += batch_count
    assert sample_count == len(dataloader.dataset)
    return {'loss': loss_sum / sample_count, 'accuracy': correct / sample_count}

@torch.inference_mode()
def evaluate(dataloader, model, loss_fn, device, collect_mistakes=False):
    model.eval()
    loss_sum, correct, sample_count = 0.0, 0, 0
    mistakes = []
    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        loss = loss_fn(logits, y)
        predicted = logits.argmax(dim=1)
        batch_count = y.size(0)
        loss_sum += loss.item() * batch_count
        correct += (predicted == y).sum().item()
        sample_count += batch_count
        if collect_mistakes and len(mistakes) < 9:
            wrong_indices = (predicted != y).nonzero(as_tuple=False).flatten().tolist()
            for index in wrong_indices[:9 - len(mistakes)]:
                mistakes.append((X[index].cpu(), y[index].item(), predicted[index].item()))
    assert sample_count == len(dataloader.dataset)
    return {'loss': loss_sum / sample_count, 'accuracy': correct / sample_count}, mistakes

## 2.13.9-训练并根据验证损失选择模型

默认执行 3 轮完整训练。短训练用于理解流程，不代表模型已充分收敛。
每轮使用训练集更新参数，然后计算验证指标；验证损失更低时保存一份独立的 CPU 权重副本。
`.clone()` 很关键，否则内存中的“最佳权重”可能随后续训练一起改变。

`history.json` 记录每轮训练与验证的损失、准确率和耗时，便于之后绘制曲线和比较实验。

In [ ]:
history = []
best_validation_loss = float('inf')
best_epoch = None
best_state = None
started = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    epoch_started = time.perf_counter()
    train_metrics = train_one_epoch(train_loader, model, loss_fn, optimizer, device)
    validation_metrics, _ = evaluate(validation_loader, model, loss_fn, device)
    record = {
        'epoch': epoch,
        'train_loss': train_metrics['loss'], 'train_accuracy': train_metrics['accuracy'],
        'validation_loss': validation_metrics['loss'], 'validation_accuracy': validation_metrics['accuracy'],
        'seconds': time.perf_counter() - epoch_started,
    }
    history.append(record)
    if validation_metrics['loss'] < best_validation_loss:
        best_validation_loss = validation_metrics['loss']
        best_epoch = epoch
        best_state = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}
    print(
        f"Epoch {epoch}/{EPOCHS} | train loss {record['train_loss']:.4f}, "
        f"acc {record['train_accuracy']:.2%} | val loss {record['validation_loss']:.4f}, "
        f"acc {record['validation_accuracy']:.2%} | {record['seconds']:.1f}s"
    )

assert best_state is not None and len(history) == EPOCHS
training_seconds = time.perf_counter() - started
model.load_state_dict(best_state)
model.eval()
WEIGHTS_PATH = RUN_DIR / 'mnist-mlp-best.state_dict.pt'
torch.save(best_state, WEIGHTS_PATH)
(RUN_DIR / 'history.json').write_text(json.dumps(history, ensure_ascii=False, indent=2), encoding='utf-8')
print('选择的 epoch:', best_epoch, '| 验证损失:', best_validation_loss)

## 2.13.10-观察训练曲线

损失越低通常表示模型对真实类别分配的概率更合理；准确率只关心最高分的类别是否正确。
训练损失持续下降而验证损失回升，可能是过拟合信号。但不能仅凭几轮训练中的波动就判断模型出现了过拟合。
根据 `history` 绘制训练曲线，虚线标出按验证损失选中的 epoch。

In [ ]:
epochs = [row['epoch'] for row in history]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs, [row['train_loss'] for row in history], marker='o', label='Train')
axes[0].plot(epochs, [row['validation_loss'] for row in history], marker='o', label='Validation')
axes[0].set_ylabel('Cross-entropy loss')
axes[1].plot(epochs, [row['train_accuracy'] for row in history], marker='o', label='Train')
axes[1].plot(epochs, [row['validation_accuracy'] for row in history], marker='o', label='Validation')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1)
for ax in axes:
    ax.axvline(best_epoch, color='gray', linestyle='--', label='Selected epoch')
    ax.set_xlabel('Epoch')
    ax.set_xticks(epochs)
    ax.grid(alpha=0.25)
    ax.legend()
fig.tight_layout()
fig.savefig(RUN_DIR / 'learning-curves.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

## 2.13.11-从 `state_dict` 恢复，并检查预测一致

权重文件保存参数与缓冲区，不保存 Python 模型定义；恢复时必须先创建相同结构。
本例只保存用于推理的权重，若要精确续训还需要优化器状态、训练进度及随机数状态等信息。

用 `map_location` 将权重映射到当前设备，显式指定 `weights_only=True`。
只加载自己生成或来源可信的文件。然后使用同一验证批次比较 logits 与预测类别：
前者用浮点容差检查，后者要求完全一致。此检查验证保存/恢复流程，不代表模型准确。

In [ ]:
restored_model = NeuralNetwork().to(device)
restored_state = torch.load(WEIGHTS_PATH, map_location=device, weights_only=True)
restored_model.load_state_dict(restored_state)
restored_model.eval()

with torch.inference_mode():
    reference_logits = model(X_example.to(device))
    restored_logits = restored_model(X_example.to(device))
torch.testing.assert_close(reference_logits, restored_logits, rtol=1e-5, atol=1e-6)
assert torch.equal(reference_logits.argmax(dim=1), restored_logits.argmax(dim=1))
print('重载检查通过：同一输入的 logits 接近，预测类别一致。')

## 2.13.12-最终测试与误分类观察

现在模型和 epoch 已经由验证集确定，再读取官方测试分区，**一次完整遍历**同时计算最终指标和收集错误样本。
标签 `True` 是真实数字，`Pred` 是预测数字；展示的是按固定测试顺序遇到的前 9 个错误，并非随机抽样或最难样本。

测试准确率只反映模型在该测试分区上的表现。手机拍摄的手写数字可能在光照、背景和倾斜程度上与 MNIST 不同，识别效果也可能变化。

In [ ]:
test_data = datasets.MNIST(root=str(DATA_ROOT), train=False, download=False, transform=ToTensor())
assert len(test_data) == 10_000
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_metrics, mistakes = evaluate(test_loader, restored_model, loss_fn, device, collect_mistakes=True)
print(f"Final test | loss {test_metrics['loss']:.4f} | accuracy {test_metrics['accuracy']:.2%}")

if mistakes:
    fig, axes = plt.subplots(3, 3, figsize=(7, 7))
    for ax in axes.flat:
        ax.axis('off')
    for ax, (sample, true_label, predicted_label) in zip(axes.flat, mistakes):
        ax.imshow(sample.squeeze(0), cmap='gray', vmin=0, vmax=1)
        ax.set_title(f'True: {true_label} | Pred: {predicted_label}')
    fig.suptitle('First misclassified test examples')
    fig.tight_layout()
    fig.savefig(RUN_DIR / 'misclassified.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
else:
    print('本次测试没有误分类样本。')

summary = {
    'versions': versions, 'seed': SEED, 'device': str(device),
    'data_source': DATA_SOURCE, 'architecture': '784-128-256-10 sigmoid MLP',
    'batch_size': BATCH_SIZE, 'epochs': EPOCHS, 'learning_rate': LEARNING_RATE,
    'optimizer': 'Adam', 'train_size': len(training_data),
    'validation_size': len(validation_data), 'test_size': len(test_data),
    'best_epoch': best_epoch, 'best_validation_loss': best_validation_loss,
    'test': test_metrics, 'training_seconds': training_seconds,
    'weights_file': WEIGHTS_PATH.name, 'reload_consistency_check': 'passed',
}
(RUN_DIR / 'run-summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print('运行记录:', RUN_DIR.relative_to(WORK_DIR) / 'run-summary.json')